# Multitask Preclinical Toxicity Prediction with GIN and Avalon Fingerprints

This notebook builds a multitask molecular toxicity classifier for the 12 endpoints in the Tox21 dataset. It combines two molecular representations: a graph representation processed by a Graph Isomorphism Network (GIN), and a 1,024-bit Avalon fingerprint processed alongside the graph embedding by a multilayer perceptron (MLP).

The notebook covers data preparation, representation generation, graph construction, masked learning with incomplete labels, hyperparameter optimization, and held-out test evaluation. 

## 1. Imports 

The notebook uses RDKit for molecular processing and Avalon fingerprints, PyTorch and PyTorch Geometric for model training, scikit-learn for data splitting and metrics, and Optuna for hyperparameter search.

In [ ]:
from rdkit import Chem
import pandas as pd 
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from rdkit.Avalon.pyAvalonTools import GetAvalonFP
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool
import optuna
from optuna.trial import TrialState


A fixed random seed is set for NumPy, Python, and PyTorch. The current configuration runs on the CPU.

In [ ]:
device = torch.device("cpu")
seed = 42
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

## 2. Load and clean the Tox21 dataset

The CSV is loaded into a pandas DataFrame. SMILES strings are stripped of surrounding whitespace and validated with RDKit. Invalid strings and molecules with one or zero atoms are removed.

In [ ]:
df = pd.read_csv("tox21.csv")

def clean_and_validate_smiles(smi):
    if not isinstance(smi, str):
        return None
    smi = smi.strip().replace(" ", "")
    if "." in smi:
        smi = smi.split(".")[0]
    mol = Chem.MolFromSmiles(smi)
    if mol is not None and mol.GetNumAtoms() > 1:
        return smi
    return None

df["smiles"] = df["smiles"].apply(clean_and_validate_smiles)
df = df[df["smiles"].notna()].reset_index(drop=True)
print("Valid SMILES:", len(df))


## 3. Generate Avalon molecular fingerprints

Each cleaned SMILES string is converted into a 1,024-bit Avalon fingerprint. The fingerprint is a fixed-length vector that encodes substructural information and provides a complementary representation to the molecular graph

In [ ]:
def get_avalon_fp(smi, n_bits=1024):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return np.zeros(n_bits, dtype=np.float32)
    fp = GetAvalonFP(mol, nBits=n_bits)
    return np.array(fp, dtype=np.float32)

smiles_list = df["smiles"].tolist()
avalon_fps = np.array([get_avalon_fp(s) for s in smiles_list], dtype=np.float32)


# 4. Define the 12 toxicity endpoints

The target matrix contains the 12 Tox21 assay labels listed in the code. Values are converted to floating point so missing labels can remain represented as NaN. The model predicts all endpoints jointly, while the loss function below ignores missing labels rather than treating them as negative examples.

In [ ]:
target_cols = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase',
    'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma',
    'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53'
]

labels = df[target_cols].values.astype(float)
num_tasks = labels.shape[1]

print("Tasks:", target_cols)
print("Number of tasks:", num_tasks)

# 5. Build  molecular graphs

Each molecule is represented as a PyTorch Geometric Data object. Node features encode atom identity, degree, formal charge, hydrogen count, aromaticity, ring membership, and hybridization. Bonds are represented as bidirectional edges; bond attributes are not included in this GIN implementation. Each graph also stores its Avalon fingerprint and the row of 12 endpoint labels.

In [ ]:
ATOM_LIST = list(range(1, 119))   # atomic numbers 1–118

def atom_features(atom):
    at_num = atom.GetAtomicNum()
    atom_onehot = [int(at_num == a) for a in ATOM_LIST]

    degree = atom.GetTotalDegree()
    degree_onehot = [int(degree == i) for i in range(0, 6)]

    formal_charge = atom.GetFormalCharge()
    fc = [formal_charge, int(formal_charge == 0)]

    num_h = atom.GetTotalNumHs()
    num_h_onehot = [int(num_h == i) for i in range(0, 5)]

    aromatic = [int(atom.GetIsAromatic())]
    in_ring = [int(atom.IsInRing())]

    hyb = atom.GetHybridization()
    hyb_onehot = [
        int(hyb == Chem.rdchem.HybridizationType.SP),
        int(hyb == Chem.rdchem.HybridizationType.SP2),
        int(hyb == Chem.rdchem.HybridizationType.SP3),
        int(hyb == Chem.rdchem.HybridizationType.SP3D),
        int(hyb == Chem.rdchem.HybridizationType.SP3D2),
    ]

    return np.array(
        atom_onehot +
        degree_onehot +
        fc +
        num_h_onehot +
        aromatic +
        in_ring +
        hyb_onehot,
        dtype=np.float32
    )


def mol_to_pyg(smiles, avalon_fp, y_row):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        return None

    # Node features
    x = [atom_features(a) for a in mol.GetAtoms()]
    x = torch.tensor(x, dtype=torch.float32)

    # Edges (no edge_attr used in this GIN; just connectivity)
    edge_index = []
    for bond in mol.GetBonds():
        a = bond.GetBeginAtomIdx()
        b = bond.GetEndAtomIdx()
        edge_index.append([a, b])
        edge_index.append([b, a])

    if len(edge_index) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    y = torch.tensor(y_row, dtype=torch.float32).unsqueeze(0)    # may contain NaNs
    fp = torch.tensor(avalon_fp, dtype=torch.float32).unsqueeze(0)

    return Data(x=x, edge_index=edge_index, y=y, fp=fp)


pyg_data_list = []
for smi, fp, y_row in zip(smiles_list, avalon_fps, labels):
    d = mol_to_pyg(smi, fp, y_row)
    if d is not None:
        pyg_data_list.append(d)

print("PyG graphs:", len(pyg_data_list))
node_dim = pyg_data_list[0].x.shape[1]
print("Node feature dim:", node_dim)


## 6. Inspect label balance and calculate positive-class weights

Label counts are calculated per endpoint, and the positive-class weight is estimated as the number of observed negatives divided by the number of observed positives. Missing (NaN) labels are excluded from these counts. The overall positive, negative, and missing-label totals are printed for a quick data-quality check.

In [ ]:
Y_all = np.vstack([d.y.numpy() for d in pyg_data_list])
task_pos = np.nansum(Y_all == 1, axis=0)
task_neg = np.nansum(Y_all == 0, axis=0)

pos_weight_np = task_neg / np.maximum(task_pos, 1)
pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32, device=device)
print("pos_weight:", pos_weight)

In [ ]:
total_pos = np.nansum(Y_all == 1)
total_neg = np.nansum(Y_all == 0)
total_nan = np.sum(np.isnan(Y_all))

print("TOTAL positives:", total_pos)
print("TOTAL negatives:", total_neg)
print("TOTAL NaNs:", total_nan)


## 7. Create training, validation, and test splits

The molecules are randomly split into training, validation, and test sets using a fixed random_state: 80% for training and 10% each for validation and testing. This is a random split, not a scaffold split, so molecules with related chemical scaffolds may occur across partitions. Data loaders batch the graph objects for model training and evaluation.

In [ ]:
indices = np.arange(len(pyg_data_list))

# Here we do a simple random split; for multi-label, stratify is tricky.
train_idx, temp_idx = train_test_split(
    indices, test_size=0.20, random_state=seed, shuffle=True
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=seed, shuffle=True
)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

def make_loader(idx_list, batch_size, shuffle):
    subset = [pyg_data_list[i] for i in idx_list]
    return DataLoader(subset, batch_size=batch_size, shuffle=shuffle)

# We'll set a default batch_size; Optuna will override for GIN model
DEFAULT_BATCH_SIZE = 32

train_loader_base = make_loader(train_idx, DEFAULT_BATCH_SIZE, shuffle=True)
val_loader_base   = make_loader(val_idx,   DEFAULT_BATCH_SIZE, shuffle=False)
test_loader_base  = make_loader(test_idx,  DEFAULT_BATCH_SIZE, shuffle=False)

## 8. Loss function and Evaluation metrics

MaskedBCEBatch computes binary cross-entropy with logits separately for each endpoint and skips missing labels. The per-endpoint positive-class weights help address label imbalance. The evaluation helpers calculate macro ROC-AUC across endpoints and aggregate accuracy, precision, recall, and F1 over observed label entries at a specified probability threshold.

In [ ]:
class MaskedBCEBatch(nn.Module):
    def __init__(self, pos_weight):
        super().__init__()
        self.pos_weight = pos_weight  # (T,)

    def forward(self, logits, targets):
        """
        logits, targets: (B, T) with NaNs in targets for missing labels
        """
        mask = ~torch.isnan(targets)          # (B, T)
        if mask.sum() == 0:
            return torch.tensor(0.0, device=logits.device)

        loss_sum = 0.0
        count = 0
        B, T = targets.shape

        for t in range(T):
            m_t = mask[:, t]
            if m_t.sum() == 0:
                continue
            logit_t = logits[m_t, t]
            target_t = targets[m_t, t]
            pw_t = self.pos_weight[t]
            loss_t = F.binary_cross_entropy_with_logits(
                logit_t, target_t, pos_weight=pw_t, reduction="mean"
            )
            loss_sum += loss_t
            count += 1

        return loss_sum / max(count, 1)

criterion = MaskedBCEBatch(pos_weight=pos_weight)

## 9. Model architecture: GIN + Avalon fingerprint fusion

The GIN encoder applies two GINConv layers to atom features and molecular connectivity, with ReLU activations and global mean pooling to obtain one graph embedding per molecule. The graph embedding is concatenated with the 1,024-bit Avalon fingerprint. An MLP then maps the combined representation to 12 output logits, one for each toxicity endpoint.

In [ ]:
def macro_auc_from_probs(y_true_np, probs_np):
    aucs = []
    T = y_true_np.shape[1]
    for t in range(T):
        mask = ~np.isnan(y_true_np[:, t])
        if mask.sum() < 10:
            continue
        y_t = y_true_np[mask, t]
        p_t = probs_np[mask, t]
        if len(np.unique(y_t)) < 2:
            continue
        try:
            auc = roc_auc_score(y_t, p_t)
        except ValueError:
            auc = np.nan
        aucs.append(auc)
    if len(aucs) == 0:
        return np.nan
    return float(np.nanmean(aucs))

def compute_macro_metrics(y_true_np, probs_np, threshold=0.5):
    mask = ~np.isnan(y_true_np)
    y_true_flat = y_true_np[mask].astype(int)
    probs_flat = probs_np[mask]
    preds_flat = (probs_flat >= threshold).astype(int)

    acc = accuracy_score(y_true_flat, preds_flat)
    prec = precision_score(y_true_flat, preds_flat, zero_division=0)
    rec = recall_score(y_true_flat, preds_flat, zero_division=0)
    f1 = f1_score(y_true_flat, preds_flat, zero_division=0)
    auc = macro_auc_from_probs(y_true_np, probs_np)
    return acc, prec, rec, f1, auc

def best_f1_threshold(y_true_np, probs_np):
    mask = ~np.isnan(y_true_np)
    y_true_flat = y_true_np[mask].astype(int)
    probs_flat = probs_np[mask]

    best_t = 0.5
    best_f1 = 0.0
    for t in np.arange(0.05, 0.95, 0.05):
        preds = (probs_flat >= t).astype(int)
        f1 = f1_score(y_true_flat, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    return best_t



In [ ]:
class GINEncoder(nn.Module):
    def __init__(self, node_dim, gin_hidden_dim=128, graph_dim = 128):
        super().__init__()
        nn1 = nn.Sequential(
            nn.Linear(node_dim, gin_hidden_dim),
            nn.ReLU(),
            nn.Linear(gin_hidden_dim, gin_hidden_dim)
        )
        self.conv1 = GINConv(nn1)

        nn2 = nn.Sequential(
            nn.Linear(gin_hidden_dim, gin_hidden_dim),
            nn.ReLU(),
            nn.Linear(gin_hidden_dim, graph_dim)
        )
        self.conv2 = GINConv(nn2)

    def forward(self, x, edge_index, batch):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.conv2(h, edge_index)
        h = F.relu(h)
        g_emb = global_mean_pool(h, batch)   # (B, out_dim)
        return g_emb


class CombinedGINAvalon(nn.Module):
    def __init__(
        self,
        node_dim,
        avalon_dim=1024,
        gin_hidden_dim=128,
        graph_dim=128,
        hidden1=512,
        hidden2=256,
        dropout=0.3,
        num_tasks=12
    ):
        super().__init__()
        self.gnn = GINEncoder(node_dim, gin_hidden_dim=gin_hidden_dim, graph_dim=graph_dim)
        self.mlp = nn.Sequential(
            nn.Linear(avalon_dim + graph_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, num_tasks)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        g_emb = self.gnn(x, edge_index, batch)   # (B, graph_dim)
        fp = data.fp                             # (B, 1024) or (1024,)
        if fp.dim() == 1:
            fp = fp.unsqueeze(0)
        z = torch.cat([fp, g_emb], dim=-1)       # (B, 1024 + graph_dim)
        return self.mlp(z)                       # (B, num_tasks)


## 10. Training and prediction helpers

The training helper performs one epoch of optimization. The evaluation helper switches the model to evaluation mode, disables gradient tracking, and returns predicted probabilities together with the corresponding labels. Sigmoid is applied to model logits to convert them into probabilities.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch)
        loss = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)

def eval_model(model, loader):
    model.eval()
    all_probs = []
    all_targets = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch)
            probs = torch.sigmoid(logits).cpu().numpy()
            y = batch.y.cpu().numpy()
            all_probs.append(probs)
            all_targets.append(y)
    probs_np = np.vstack(all_probs)
    y_np = np.vstack(all_targets)
    return y_np, probs_np


## 11. Hyperparameter optimization with Optuna

Optuna searches over GIN dimensions, MLP hidden-layer sizes, dropout, learning rate, weight decay, batch size, and epoch count. Each trial trains on the training split and is scored using validation macro ROC-AUC; the study is configured to maximize this score. The best trial parameters are then used to initialize the final model.

In [ ]:
def objective(trial):
    gin_hidden_dim = trial.suggest_categorical("gin_hidden_dim", [64, 128, 192])
    graph_dim      = trial.suggest_categorical("graph_dim",     [64, 128, 192])
    hidden1        = trial.suggest_categorical("hidden1",       [256, 512, 768, 1024])
    hidden2        = trial.suggest_categorical("hidden2",       [128, 256, 384, 512])
    dropout        = trial.suggest_float("dropout", 0.1, 0.5)
    lr             = trial.suggest_float("lr", 1e-4, 2e-3, log=True)
    weight_decay   = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    batch_size     = trial.suggest_categorical("batch_size", [16, 32, 64])
    epochs         = trial.suggest_int("epochs", 20, 40)

    # Build loaders with this batch_size
    train_loader = make_loader(train_idx, batch_size, shuffle=True)
    val_loader   = make_loader(val_idx,   batch_size, shuffle=False)

    model = CombinedGINAvalon(
        node_dim=node_dim,
            avalon_dim=1024,
            gin_hidden_dim=gin_hidden_dim,
            graph_dim=graph_dim,
            hidden1=hidden1,
            hidden2=hidden2,
            dropout=dropout,
            num_tasks=num_tasks
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )

    # Train
    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)

    # Evaluate on validation set
    y_val_np, val_probs_np = eval_model(model, val_loader)
    val_auc = macro_auc_from_probs(y_val_np, val_probs_np)
    # Optuna tries to maximize
    return val_auc

# Run Optuna study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=25)

print("Number of finished trials:", len(study.trials))
print("Best value (Val Macro AUC):", study.best_value)
print("Best params:", study.best_params)

best_params = study.best_params


## 12. Final training and test set evaluation 

The final model is trained for 50 epochs on the combined training and validation data, using the hyperparameters selected by Optuna. Predictions are then generated for the held-out test split, and accuracy, precision, recall, F1, and macro ROC-AUC are printed.

In [ ]:
gin_model = CombinedGINAvalon(
        node_dim=node_dim,
        avalon_dim=1024,
        gin_hidden_dim=best_params["gin_hidden_dim"],
        graph_dim=best_params["graph_dim"],
        hidden1=best_params["hidden1"],
        hidden2=best_params["hidden2"],
        dropout=best_params["dropout"],
        num_tasks=num_tasks
    ).to(device)

gin_optimizer = torch.optim.Adam(
    gin_model.parameters(),
    lr=best_params["lr"],
    weight_decay=best_params["weight_decay"],
)

# Train+Val combined indices
trainval_idx = np.concatenate([train_idx, val_idx])
trainval_loader = make_loader(trainval_idx, best_params["batch_size"], shuffle=True)
test_loader = make_loader(test_idx, best_params["batch_size"], shuffle=False)

EPOCHS_FINAL_GIN = 50
losses = []
for epoch in range(EPOCHS_FINAL_GIN):
    loss = train_one_epoch(gin_model, trainval_loader, gin_optimizer, criterion)
    losses.append(loss)
    if (epoch + 1) % 10 == 0:
        print(f"[GIN+FP] Epoch {epoch+1}/{EPOCHS_FINAL_GIN}, Loss = {loss:.4f}")

# Evaluate on test
y_test_np_gin, test_probs_np_gin = eval_model(gin_model, test_loader)
best_t_gin = best_f1_threshold(y_test_np_gin, test_probs_np_gin)
acc_gin, prec_gin, rec_gin, f1_gin, auc_gin = compute_macro_metrics(
    y_test_np_gin, test_probs_np_gin, threshold=best_t_gin
)

print("\n===== GIN + Avalon FP (Test) =====")

print(f"Accuracy : {acc_gin:.4f}")
print(f"Precision: {prec_gin:.4f}")
print(f"Recall   : {rec_gin:.4f}")
print(f"F1       : {f1_gin:.4f}")
print(f"ROC-AUC  : {auc_gin:.4f}")